# Grafici su settore estero

Questa categoria genera o registra grafici su import, export, quote mondiali, partite correnti, NIIP e ragioni di scambio. Alcuni dati sono disponibili sia in World Bank sia in IMF/OECD/Eurostat.

Ogni grafico riporta la fonte primaria usata nella generazione e la dicitura `elaborazione Nazareno Lecis`.


## Come vengono generati

Il notebook separa tre cose: la specifica del grafico, la fonte dati e la trasformazione. Ogni specifica dichiara `titolo`, `tipo_grafico`, `anno_inizio`, `ultimo_dato`, fonte primaria, fonti alternative e cosa viene mostrato. `definisci_grafico_da_indicatore_world_bank` usa direttamente un indicatore WDI; `definisci_grafico_da_rapporto_world_bank` combina due serie WDI; `registra_grafico_da_collegare_a_api` mantiene in inventario un grafico per cui la fonte pubblica esiste ma non e ancora mappata nel codice.

Quando il grafico viene generato, `genera_grafici_e_inventario` scarica i dati via API, applica la trasformazione indicata, costruisce il confronto con Italia, min-max, percentili 25-75 e mediana dei paesi avanzati quando disponibile, poi mostra l'inventario e lascia il plot nel notebook.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "analisi").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analisi.utils.grafici import (
    PAESI_MONDO,
    definisci_grafico_da_rapporto_world_bank,
    definisci_grafico_da_indicatore_world_bank,
    genera_grafici_e_inventario,
    registra_grafico_da_collegare_a_api,
)

from sources.api_catalog import FONTI_API


## Fonti disponibili per questa categoria

La tabella sotto elenca le API ufficiali considerate per la categoria. Una stessa variabile puo comparire in piu fonti: il notebook indica una fonte primaria per la generazione, ma mantiene le alternative quando esistono definizioni ufficiali equivalenti o vicine.


In [ ]:
fonti_categoria = ('World Bank API', 'IMF DataMapper API', 'Eurostat API', 'OECD SDMX API')
catalogo_fonti_api = pd.DataFrame(
    [
        {
            "fonte": fonte.nome,
            "ente": fonte.ente,
            "aree": ", ".join(fonte.aree),
            "note": fonte.note,
            "endpoint_controllo": fonte.url_controllo,
        }
        for fonte in FONTI_API
        if fonte.nome in fonti_categoria
    ]
)
catalogo_fonti_api


## Specifiche dei grafici

Le righe sotto sono volutamente esplicite: per ogni grafico si legge il titolo dell'analisi, il tipo di grafico, la fonte primaria, le fonti ufficiali alternative, l'anno di partenza, l'ultimo dato richiesto e la trasformazione applicata.


In [ ]:
SPECIFICHE_GRAFICI = [
    definisci_grafico_da_indicatore_world_bank(
        titolo='Import reale - crescita media',
        nome_output='import_reale_crescita_media',
        indicatore='NE.IMP.GNFS.KD',
        trasformazione='growth_10y',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Import reale - crescita media: tasso di crescita annualizzato su 10 anni, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Import reale - crescita cumulata',
        nome_output='import_reale_crescita_cumulata',
        indicatore='NE.IMP.GNFS.KD',
        trasformazione='index',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Import reale - crescita cumulata: indice con base 1970=100, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Export reale - crescita media',
        nome_output='export_reale_crescita_media',
        indicatore='NE.EXP.GNFS.KD',
        trasformazione='growth_10y',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Export reale - crescita media: tasso di crescita annualizzato su 10 anni, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Export reale - crescita cumulata',
        nome_output='export_reale_crescita_cumulata',
        indicatore='NE.EXP.GNFS.KD',
        trasformazione='index',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Export reale - crescita cumulata: indice con base 1970=100, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Esportazioni di beni e servizi - livello',
        nome_output='esportazioni_beni_servizi_livello',
        indicatore='NE.EXP.GNFS.CD',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Esportazioni di beni e servizi - livello: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Esportazioni nette e saldo commerciale reale',
        nome_output='saldo_commerciale_reale',
        fonte_primaria='World Bank API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Esportazioni nette e saldo commerciale reale: Richiede saldo reale coerente per beni, servizi e totale.',
        fonti_alternative=('IMF API', 'IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
        note='Richiede saldo reale coerente per beni, servizi e totale.',
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo="Esportazioni - quota dell'export mondiale",
        nome_output='quota_export_mondiale',
        indicatore='NE.EXP.GNFS.CD',
        trasformazione='share_world',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra="Esportazioni - quota dell'export mondiale: quota dell'Italia sul totale mondiale, senza distribuzione dei paesi avanzati.",
        fonti_alternative=('IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
        confronto=False,
        paesi=tuple(PAESI_MONDO),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo="Importazioni - quota dell'import mondiale",
        nome_output='quota_import_mondiale',
        indicatore='NE.IMP.GNFS.CD',
        trasformazione='share_world',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra="Importazioni - quota dell'import mondiale: quota dell'Italia sul totale mondiale, senza distribuzione dei paesi avanzati.",
        fonti_alternative=('IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
        confronto=False,
        paesi=tuple(PAESI_MONDO),
    ),
    registra_grafico_da_collegare_a_api(
        titolo="Commercio mondiale - quota dell'Italia",
        nome_output='quota_commercio_mondiale',
        fonte_primaria='World Bank API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra="Commercio mondiale - quota dell'Italia: Richiede somma export+import dell'Italia rapportata alla somma mondiale.",
        fonti_alternative=('IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
        note="Richiede somma export+import dell'Italia rapportata alla somma mondiale.",
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Esportazioni relative di beni e servizi',
        nome_output='esportazioni_relative_beni_servizi',
        fonte_primaria='World Bank API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Esportazioni relative di beni e servizi: Richiede rapporto con paniere di paesi o mondo.',
        fonti_alternative=('OECD SDMX API', 'IMF BOP API', 'Eurostat API'),
        note='Richiede rapporto con paniere di paesi o mondo.',
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Partite correnti - % del PIL',
        nome_output='partite_correnti_pil',
        indicatore='BN.CAB.XOKA.GD.ZS',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Partite correnti - % del PIL: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Posizione internazionale netta',
        nome_output='posizione_internazionale_netta',
        fonte_primaria='IMF IFS API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Posizione internazionale netta: NIIP da collegare via fonte internazionale ufficiale.',
        fonti_alternative=('IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
        note='NIIP da collegare via fonte internazionale ufficiale.',
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Ragioni di scambio merci',
        nome_output='ragioni_scambio_merci',
        indicatore='TT.PRI.MRCH.XD.WD',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Ragioni di scambio merci: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('IMF BOP API', 'OECD SDMX API', 'Eurostat API'),
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Ragioni di scambio beni e servizi',
        nome_output='ragioni_scambio_beni_servizi',
        fonte_primaria='OECD SDMX API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Ragioni di scambio beni e servizi: Livello e crescita media per beni, servizi e totale.',
        fonti_alternative=('Eurostat API', 'IMF BOP API'),
        note='Livello e crescita media per beni, servizi e totale.',
    ),
]

print(f"Specifiche nella categoria: {len(SPECIFICHE_GRAFICI)}")


## Inventario e generazione

La tabella risultante mostra cosa viene prodotto, quale fonte e stata usata, quali alternative sono possibili, da quale anno parte la serie, fino a quale ultimo dato viene richiesto e quali grafici restano da collegare.


In [ ]:
cartella_output = ROOT / "analisi" / "settore_estero"
percorsi, inventario = genera_grafici_e_inventario(SPECIFICHE_GRAFICI, cartella_output)

colonne = [
    "titolo",
    "tipo_grafico",
    "cosa_mostra",
    "fonte_primaria",
    "fonti_alternative",
    "anno_inizio",
    "ultimo_dato",
    "trasformazione",
    "confronto",
    "stato",
    "nome_output",
    "note",
    "errore",
]
print(f"PNG generati: {len(percorsi)}")
for percorso in percorsi:
    print(percorso)
inventario[colonne]


## Plot dei grafici generati

I plot visualizzati qui sono quelli creati dalla cella precedente. I PNG locali restano fuori da Git.


In [ ]:
from IPython.display import Image, display

if not percorsi:
    print("Nessun PNG generato: tutti i grafici della categoria sono ancora da collegare o la fonte non ha restituito dati.")
for percorso in percorsi:
    display(Image(filename=str(percorso)))
